# 从全连接层到CNN

## 多层感知机的限制

对于多层感知机(以单层为例)：
$$
H = XW^{(1)} + b^{(1)} \\
O = HW^{(2)} + b^{(2)}
$$

对于输入为图片，一次输入会有至少100万个维度（百万像素级），隐藏层维度即使降到1000，这个全连接层也会保存至少$10^{9}$个参数

卷积层的本质是把输入与kernel矩阵进行交叉相关，再加上偏置之后得到的输出，kernel和b是可以学习到的参数

或者说卷积层实际上就是引入了平移不变性和局部性的全连接方式

## 重新考察全连接层

* 输入和输出从向量变成了矩阵（图片，不再展平）
* 权重变成4-D张量（原来的输入输出是一维， 权重矩阵即为2维，现在输入输出变成二维，权重表示输入输出的下标即为四维）

$$
h_{ij} = \sum_{k, l} w_{i,j,k,l}x_{k,l}
$$

ij 表示索引的位置

### 平移不变性

平移不变性的意思是，输入x的ij无论怎么变化，v都不会发生变化，即v不会依赖ij

在上面的公式中， x位置的平移会导致h的平移，$h_{ij} = \sum_{k, l} w_{i,j,k,l} \times x_{k,l} = \sum_{a, b} v_{i,j,a,b} \times x_{i+a, j+b} $

综合有：$v_{i,j,a,b} \rightarrow v_{a,b}$

即
$$
h_{i,j} = \sum_{a, b} v_{a, b}x_{i+a, j+b}
$$

### 局部性

局部性是指在评估h_{i, j}的时候，只关注局部的信息，不应该用原理x_{i, j}的参数；

解决方案是设置一个阈值$\delta$, 当$|a|, |b| \geq \delta$ 时，$v_{a, b} = 0$

写成公式就是：
$$
h_{i, j} = \sum_{a = -\delta}^{\delta} \sum_{b = -\delta}^{\delta} v_{a, b}x_{i+a, j+b}
$$

对全连接层使用平移不变性和局部性即可得到卷积层

## 卷积

时域上的卷积公式：
$$
F(t) = f(t) \ast g(t) = \int_{-\infty}^{\infty} f(\tau)g(t-\tau) d\tau
$$

连续时间卷积的一个比较狭隘的理解是，针对不确定的输入信号和确定的响应系统，去求取t时刻系统的存量

图像卷积的本质是：
* 过去对现在的影响
* 周围一圈像素点对当前像素点的影响

## 互相关运算
其实卷积层本质上是一个错误的叫法，他所表达的运算实际上是一种互相关运算

* 输入X: $n_k  \times n_w $
* kernel: $k_n \times k_w$
* bias: $b \in \mathbb{R}$
* output $(n_h - k_h + 1) \times (n_w - k_w + 1)$

这样的互相关运算和卷积公式非常相似，我们已二维互相关和二维交叉运算为例
$$
y_{i, j} = \sum_{a=1}^{h} \sum_{b=1}^{w} w_{a, b}x_{i+a}{j+b} (互相关) \\
y_{i, j} = \sum_{a=1}^{h} \sum_{b=1}^{w} w_{-a, -b}x_{i+a}{j+b} (卷积运算)
$$

实际上互相关就是把kernel翻转180度，再进行hardmard积

## 代码的基本实现 

### 实现二维的互相关计算

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(X, K):  #@save
    """计算二维互相关运算"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


测试一下

In [2]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

这里可以看到，直接进行卷积是会降维的，所以后面需要进行填充

### 实现卷积层

In [4]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))  # 这里是看传入的kernel_size是几维的
        self.bias = nn.Parameter(torch.zeros(1))  # 偏置项置为0

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

### 实现图像中目标的边缘检测

In [6]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X  # 构造一个6 * 8的矩阵，中间4列是0（黑色），其他列是1（白色）

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

构造一个高度为1，宽度为2的卷积核K，当进行互相关运算的时候，如果水平相邻的两个元素相同，则输出位0否则输出即为非0

In [7]:
K = torch.tensor([[1.0, -1.0]])
K
Y = corr2d(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

这里可以看到，对参数检测出了图像的垂直边缘，但是翻转之后失效了，说明无法检测出水平边缘

In [8]:
corr2d(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

### 学习卷积核

In [9]:
# 构造一个二维卷积层，它具有1个输出通道和形状为（1，2）的卷积核
conv2d = nn.Conv2d(1,1, kernel_size=(1, 2), bias=False)

# 这个二维卷积层使用四维输入和输出格式（批量大小、通道、高度、宽度），
# 其中批量大小和通道数都为1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))  # 这里继承了前面的X，Y
lr = 3e-2  # 学习率

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()  # 更新y_hat的梯度，即卷积核的梯度
    # 迭代卷积核
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

epoch 2, loss 5.935
epoch 4, loss 1.007
epoch 6, loss 0.174
epoch 8, loss 0.031
epoch 10, loss 0.006


查看学习到的权重张量

In [10]:
conv2d.weight.data.reshape((1, 2))

tensor([[ 0.9837, -0.9911]])